# Fugacity ideal-model benchmark explorer

Run a Google Benchmark executable (e.g. `build/release/benchmarks/bench_ideal_models`),
capture its output, parse it, and explore the results interactively.

**What you get**

- A run cell that executes the benchmark, saving a human-readable `.txt` and a
  machine-readable `.json` to `./results/`.
- A parser that turns the JSON into a tidy `pandas.DataFrame`.
- An interactive plot:
  - dropdowns to pick the **calculation** (helmholtz, pressure, ...) and the **model**
    (Nasa7, ConstantCp);
  - **x-axis** = `N` (number of species), **y-axis** = CPU time;
  - two lines — *Static* (compile-time size) and *Dynamic* (runtime size);
  - click any legend entry to toggle that series on/off.

> **Kernel:** select **Fugacity Bench (.venv)**. If it is missing, run from the repo root:
> `benchmarks/notebook/.venv/bin/python -m ipykernel install --user --name fugacity-bench --display-name "Fugacity Bench (.venv)"`


In [ ]:
from pathlib import Path
import json, re, subprocess, sys
import pandas as pd

# --- Configuration ---------------------------------------------------------
# Locate the repo root by walking up to the git top level (the benchmarks/
# subdirectory has its own CMakeLists.txt, so we key on the .git marker).
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / ".git").exists():
    REPO_ROOT = REPO_ROOT.parent

# Path to the benchmark executable. Edit this if your build tree differs.
EXE_PATH = REPO_ROOT / "build" / "release-max" / "benchmarks" / "bench_ideal_models"

# Where to write the captured output.
OUT_DIR  = Path.cwd() / "results"
OUT_DIR.mkdir(exist_ok=True)
TXT_PATH  = OUT_DIR / "bench_output.txt"     # console output (human-readable)
JSON_PATH = OUT_DIR / "bench_results.json"   # structured output (parsed below)

# Google Benchmark minimum wall-time per benchmark. Raise for steadier numbers
# (e.g. "0.2s"); lower for a quick smoke run (e.g. "0.01s").
MIN_TIME = "0.01s"

# Re-run the executable, or reuse an existing capture in ./results/ ?
RUN_BENCHMARK = True

print("repo root :", REPO_ROOT)
print("executable:", EXE_PATH, "(exists)" if EXE_PATH.exists() else "(MISSING - build it first)")
print("results   :", OUT_DIR)

In [ ]:
def run_benchmark(exe, json_path, txt_path, min_time="0.05s", extra_args=None):
    """Run the benchmark once and capture both a text and a JSON file.

    The custom driver in bench_ideal_models prints grouped banners by default, but
    *truncates* the --benchmark_out file once per block. Passing --benchmark_filter
    takes its single-run path instead, which writes one complete JSON file. We use
    `--benchmark_filter=.` (matches everything) to get every benchmark in one file.
    """
    exe = Path(exe)
    if not exe.exists():
        raise FileNotFoundError(
            f"Benchmark executable not found: {exe}\n"
            "Build it first, for example:\n"
            "  cmake --preset release -DBUILD_BENCHMARKS=ON\n"
            "  cmake --build build/release --target bench_ideal_models")
    args = [str(exe),
            "--benchmark_filter=.",
            f"--benchmark_min_time={min_time}",
            f"--benchmark_out={json_path}",
            "--benchmark_out_format=json"]
    if extra_args:
        args += list(extra_args)
    print("running:", " ".join(args), "\n")
    proc = subprocess.run(args, capture_output=True, text=True)
    txt_path.write_text(proc.stdout + (("\n" + proc.stderr) if proc.stderr else ""))
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr, file=sys.stderr)
        raise RuntimeError(f"benchmark exited with code {proc.returncode}")
    print(f"console output -> {txt_path}")
    print(f"json results   -> {json_path}")

if RUN_BENCHMARK:
    run_benchmark(EXE_PATH, JSON_PATH, TXT_PATH, MIN_TIME)
else:
    if not JSON_PATH.exists():
        raise FileNotFoundError(f"{JSON_PATH} not found - set RUN_BENCHMARK = True to create it.")
    print("reusing existing capture:", JSON_PATH)

In [ ]:
# Benchmark names look like:  Static/N10/Nasa7/chemical_potential
NAME_RE = re.compile(
    r"^(?P<phase>Static|Dynamic)/N(?P<n>\d+)/(?P<family>[A-Za-z0-9]+)/(?P<calc>.+)$")

# Convert any Google Benchmark time unit to nanoseconds.
_UNIT_NS = {"ns": 1.0, "us": 1e3, "ms": 1e6, "s": 1e9}

def parse_results(json_path):
    data = json.loads(Path(json_path).read_text())
    rows = []
    for b in data.get("benchmarks", []):
        if b.get("run_type") == "aggregate":     # skip mean/median/stddev rows
            continue
        m = NAME_RE.match(b["name"])
        if not m:
            continue
        scale = _UNIT_NS.get(b["time_unit"], 1.0)
        rows.append(dict(
            phase=m["phase"], N=int(m["n"]), family=m["family"], calc=m["calc"],
            cpu_ns=b["cpu_time"] * scale, real_ns=b["real_time"] * scale,
            raw_name=b["name"]))
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError("No benchmark rows parsed - check the executable / JSON.")
    return df.sort_values(["calc", "family", "phase", "N"]).reset_index(drop=True)

df = parse_results(JSON_PATH)
print(f"parsed {len(df)} rows")
print("calculations:", ", ".join(sorted(df["calc"].unique())))
print("models      :", ", ".join(sorted(df["family"].unique())))
print("sizes (N)   :", sorted(df["N"].unique()))
df.head(8)

## Interactive plot

Pick a **calculation** and a **model**. The *Static* (compile-time size) and *Dynamic*
(runtime size) variants are drawn as two colored lines over `N`. Use the **log**
toggles for the usual scientific log–log view, and click legend entries to isolate
series.

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# --- styling ---------------------------------------------------------------
PHASE_ORDER = ["Static", "Dynamic"]
PHASE_COLOR = {"Static": "#1f77b4", "Dynamic": "#d62728"}

# --- controls --------------------------------------------------------------
_style = {"description_width": "initial"}
calc_dd   = widgets.Dropdown(options=sorted(df["calc"].unique()),   description="Calculation:", style=_style)
family_dd = widgets.Dropdown(options=sorted(df["family"].unique()), description="Model:",       style=_style)
metric_dd = widgets.Dropdown(options=[("CPU time", "cpu_ns"), ("Wall time", "real_ns")],
                             value="cpu_ns", description="Metric:", style=_style)
logx_cb = widgets.Checkbox(value=True, description="log N")
logy_cb = widgets.Checkbox(value=True, description="log time")

fig = go.FigureWidget()
fig.update_layout(template="simple_white", width=860, height=560,
                  legend_title_text="series", hovermode="x unified",
                  margin=dict(l=75, r=30, t=70, b=60))

def _update(*_):
    calc, family, ycol = calc_dd.value, family_dd.value, metric_dd.value
    sub = df[(df["calc"] == calc) & (df["family"] == family)]
    ylabel = "CPU time [ns]" if ycol == "cpu_ns" else "Wall time [ns]"
    with fig.batch_update():
        fig.data = []
        for phase in PHASE_ORDER:
            s = sub[sub["phase"] == phase].sort_values("N")
            if s.empty:
                continue
            fig.add_scatter(
                x=s["N"], y=s[ycol], mode="lines+markers", name=phase,
                line=dict(color=PHASE_COLOR[phase], width=2),
                marker=dict(size=7),
                hovertemplate=phase + ": %{y:.4g} ns<extra></extra>")
        fig.layout.title.text = f"{calc}  —  {family}"
        fig.layout.xaxis.title.text = "N  (number of species)"
        fig.layout.yaxis.title.text = ylabel
        fig.layout.xaxis.type = "log" if logx_cb.value else "linear"
        fig.layout.yaxis.type = "log" if logy_cb.value else "linear"

for _w in (calc_dd, family_dd, metric_dd, logx_cb, logy_cb):
    _w.observe(_update, names="value")
_update()

display(widgets.VBox([widgets.HBox([calc_dd, family_dd, metric_dd]),
                      widgets.HBox([logx_cb, logy_cb]),
                      fig]))

### Static snapshot (optional)

`FigureWidget` is fully interactive, but it only renders in a live kernel. Run the
cell below to also produce a self-contained comparison grid (one subplot per
calculation for the selected model) that survives notebook export to HTML/PDF.

In [ ]:
import math
from plotly.subplots import make_subplots

def overview(model, ycol="cpu_ns", logx=True, logy=True):
    calcs = sorted(df[df["family"] == model]["calc"].unique())
    ncols = 3
    nrows = math.ceil(len(calcs) / ncols)
    fig2 = make_subplots(rows=nrows, cols=ncols, subplot_titles=calcs,
                         horizontal_spacing=0.07, vertical_spacing=0.10)
    seen = set()
    for k, calc in enumerate(calcs):
        r, c = k // ncols + 1, k % ncols + 1
        sub = df[(df["family"] == model) & (df["calc"] == calc)]
        for phase in PHASE_ORDER:
            s = sub[sub["phase"] == phase].sort_values("N")
            if s.empty:
                continue
            fig2.add_scatter(
                x=s["N"], y=s[ycol], mode="lines+markers", name=phase,
                legendgroup=phase, showlegend=phase not in seen,
                line=dict(color=PHASE_COLOR[phase], width=1.8),
                marker=dict(size=5), row=r, col=c)
            seen.add(phase)
        fig2.update_xaxes(type="log" if logx else "linear", title_text="N", row=r, col=c)
        fig2.update_yaxes(type="log" if logy else "linear", row=r, col=c)
    fig2.update_layout(template="simple_white", height=300 * nrows, width=1100,
                      title_text=f"All calculations — {model}  ({'CPU' if ycol=='cpu_ns' else 'wall'} time, ns)",
                      legend_title_text="series", margin=dict(t=90))
    return fig2

overview("Nasa7")